# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv('clientes_ecommerce.csv')
# TODO: mostrar 10 primeras filas
df.head(10)
# TODO: consultar shape, dtypes, info y número de duplicados
print(df.shape)
print(df.dtypes)
print(df.info())
print("Filas duplicadas: ",df.duplicated().sum())

(400, 12)
cliente_id              int64
edad                    int64
antiguedad_meses      float64
ingresos_anuales      float64
gasto_mensual         float64
num_compras             int64
devoluciones            int64
dispositivo               str
zona                      str
nivel_fidelidad           str
satisfaccion            int64
compra_proximo_mes        str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona                400 non

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- Hay un total de 400 muestras y 12 columnas.
- El target, es la variable **compra_proximo_mes**
- La columna "cliente_id"
- No, no hay filas duplicadas.

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [2]:
# TODO: ausentes por columna
ausentes = df.isna().sum()
print(ausentes)
# TODO: porcentaje de ausentes por columna
porcentaje = df.isna().mean().mul(100)
print(porcentaje)
# TODO: filas con algún valor ausente
# df.isna() devuelve el Matriz entero indicando True en todos los campos que sean NAN
# Any busca cualquier valor en las filas que tenga True y después lo suma
filas_incompletas = df.isna().any(axis=1).sum()
print(filas_incompletas)

cliente_id            0
edad                  0
antiguedad_meses      5
ingresos_anuales      4
gasto_mensual         3
num_compras           0
devoluciones          0
dispositivo           0
zona                  0
nivel_fidelidad       0
satisfaccion          0
compra_proximo_mes    0
dtype: int64
cliente_id            0.00
edad                  0.00
antiguedad_meses      1.25
ingresos_anuales      1.00
gasto_mensual         0.75
num_compras           0.00
devoluciones          0.00
dispositivo           0.00
zona                  0.00
nivel_fidelidad       0.00
satisfaccion          0.00
compra_proximo_mes    0.00
dtype: float64
12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1.25 |
| `ingresos_anuales` | 4 | 1.00 |
| `gasto_mensual` | 3 | 0.75 |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No, porque nos sirven igualmente para entrenar al modelo; pero tendremos que hacer una normalización de esos datos posteriormente.

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [ ]:
# TODO: mínimo y máximo de edad
print("Edad mínima: ",df['edad'].min())
print("Edad máxima: ",df['edad'].max())
# TODO: localizar registros con edad > 100

df.loc[df["edad"] > 100, ["cliente_id", "edad"]]

Edad mínima:  18
Edad máxima:  195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta: Porque ha podido ser un error del usuario, pero todos sus datos están correctos y la edad simplemente puede haber sido un fallo del sistema o humano al introducirlo;
            Pero nos interesa analizarlo.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | Gasto Mensual |
| Categóricas nominales | Dispositivo |
| Categóricas ordinales | Nivel de Fidelidad |
| Identificador | Cliente_id |
| Target | Compra Proximo Mes |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta: 
- Porque las etiquetas dentro de zona no tienen ningún orden por lo tanto es nominal; y nivel_fidelidad es ordinal ya que damos un orden de mayor a menor importancia.
- No incluiría cliente_id porque no nos aporta información importante para nuestros análisis, simplemente identifica que cliente es, pero no nos interesa.

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [22]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=['cliente_id', 'compra_proximo_mes'])
# TODO: y = ... (mapear No:0, Sí:1)
y = df['compra_proximo_mes'].map({"No":0,"Sí":1})

# TODO: X.shape, y.shape, y.value_counts()
print(X.shape)
print(y.shape)
print(y.value_counts())

(400, 10)
(400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: Hay un total de 10 que podemos verlo en el número de columnas del X.shape

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [5]:
# TODO: completar
# En random_state se utiliza una semilla que es para generar un valor aleatorio, la semilla es el punto desde donde empieza a generar
# Por lo que random_state indica desde que dato del Dataset empieza a entrenar y testear
# Stratify sirve para conservar las proporciones de Train y Test que le hemos dado en test_size para que haya el mismo porcentaje de datos en train y en test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# TODO: mostrar las dimensiones de los cuatro conjuntos
print("Entrenamiento: ",X_train.shape, y_train.shape)
print("Test: ",X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Entrenamiento:  (320, 10) (320,)
Test:  (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: 
- En el conjunto de Train hay 320 muestras y en el de Test 80.
- Stratify sirve para conservar las proporciones de Train y Test que le hemos dado en test_size para que haya el mismo porcentaje de datos en train y en test.
- Dividimos para obtener valores que pueda entender mejor el modelo que posteriormente entrenaremos.

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.
- Imputación sirve para darle un valor a los campos NaN.
- Técnicas de Imputación:
    - Media --> sustituir el valor NaN por la media de los datos de esa columna.
    - Mediana --> sustituir el valor NaN por la mediana --> **Interesante al tener Outliers**
    - Moda (valor que + se repite)
    - Valor constante --> Definimos un valor y siempre que encontramos NaN lo sustituimos por ese valor.

- Codificación (Encoding): Transformar datos de una forma a otra
    - One Hot-Encoding: Codificar valores categóricos de forma numérica
    - Ordinal: Darle una orden (nivel de fidelidad por ejemplo)

- Escalado:
    - Normalización Min-Max: Convertir un valor en otro que esté dentro del rango [0-1] 
        - Formula --> X = (X - Xmin)/(Xmax - Xmin)
    - Z-Score: Modificar los datos para conseguir una media de 0 y una desviación típica de 1
        - Fórmula --> (X - u)/o | u = media, o = desviación típica
    

In [6]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
# Hacemos que todos los valores de Train de la columna edad que sean mayor a 100 los sustituya por NaN con np.nan
print(X_train.loc[df['edad']>100,["edad"]])
X_train.loc[df['edad']>100,["edad"]] = np.nan
print(X_train['edad'].isna().sum())

# Y lo mismo con los valores de test
print(X_test.loc[df['edad']>100,['edad']])
X_test.loc[df['edad']>100,["edad"]] = np.nan
print(X_test['edad'].isna().sum())

# TODO: contar ausentes en edad en entrenamiento y prueba


     edad
137   195
1
Empty DataFrame
Columns: [edad]
Index: []
0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [7]:
# TODO: imputer_numerico = SimpleImputer(...)
""" 
strategy=
- Media = mean
- Mediana = median
- Moda = most_frequent
- Valor constante = constant => fill_value=x
"""
#imputer_numerico = SimpleImputer(strategy="constant",fill_value=20)
imputer_numerico = SimpleImputer(strategy="median")
# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy="most_frequent")

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | Mediana | Porque es una variable númerica y como puede haber outliers y la media nos da un valor más razonable |
| antiguedad_meses | Mediana | "" |
| ingresos_anuales | Mediana | "" |
| gasto_mensual | Mediana | "" |
| dispositivo | Moda | Porque es una variable categorica |
| zona | Moda | "" |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: Porque podríamos tener una fuga de datos. Y queremos solo aplicarlo a los datos de entrenamiento para posteriormente aplicar "transform" a los datos de test.

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [8]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
# handle_unknown="ignore" => Si encuentra una categoria desconocida la ignora 
# sparse_output=False => El resultado me devuelva un array de numpy
encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
# TODO: aplicar fit_transform a ejemplo
matriz_ejemplo = encoder_nominal.fit_transform(ejemplo)
# - fit() => Localiza las categoría de forma única -> [Movil, ordenador, tablet]
# - transform() => Genera el array con eso datos
matriz_ejemplo
# TODO: consultar get_feature_names_out()
# La función genera el nombre de la columna a partir de los datos que le hemos pasado
pd.DataFrame(matriz_ejemplo,columns=encoder_nominal.get_feature_names_out())

,dispositivo_Móvil,dispositivo_Ordenador,dispositivo_Tablet
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: 
- Se generan 3 columnas.
- Representan cada tipo de dato que hay dentro de la "matriz_ejemplo".
- Por qué al entrenar el modelo, podría darle más peso a un valor o más importancia a uno que a otro. Y no queremos eso ya que no hay ningún orden o jerarquía.

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [9]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[...]])
encoder_ordinal = OrdinalEncoder(
    categories=[["Bronce","Plata","Oro","Platino"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1
    )
# TODO: transformar ejemplo_fidelidad y mostrar el resultado
matriz_ordinal = encoder_ordinal.fit_transform(ejemplo_fidelidad)

pd.DataFrame({"nivel":ejemplo_fidelidad['nivel_fidelidad'],"codigo":matriz_ordinal.ravel()})
# Añade el resultado directamente como una nueva columna
ejemplo_fidelidad["nivel_fidelidad_codificado"] = matriz_ordinal

# Muestra el DataFrame resultante
print(ejemplo_fidelidad)
#


  nivel_fidelidad  nivel_fidelidad_codificado
0             Oro                         2.0
1          Bronce                         0.0
2         Platino                         3.0
3           Plata                         1.0


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta:
- Los códigos representan el nivel de jerarquía de esa variable para que el modelo aprenda cual tiene mayor peso.
- Sí, es exactamente 1.

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [10]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# TODO: crear MinMaxScaler y StandardScaler
# Min-Max define valores entre 0 y 1 y Z-Score valores desde el 0 hacia arriba o hacia abajo
minmax = MinMaxScaler()
zscore = StandardScaler()
# TODO: transformar datos con cada uno
minmax.fit_transform(datos)
# Zscore genera valores tomando al valor de la mediana como 0 y a lo que esté antes < 0 y después > 0
zscore.fit_transform(datos)

# TODO: mostrar tabla comparativa
# Visualización de los datos transformados
comparacion = {
    "datos":datos['valor'],
    "minmax":minmax.fit_transform(datos).ravel(),
    "zscore":zscore.fit_transform(datos).ravel()
    }
print(pd.DataFrame(comparacion))

   datos  minmax    zscore
0     10    0.00 -1.414214
1     20    0.25 -0.707107
2     30    0.50  0.000000
3     40    0.75  0.707107
4     50    1.00  1.414214


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.00 | -1.414214 |
| 20 | 0.25 | -0.707107 |
| 30 | 0.50 | 0.000000 |
| 40 | 0.75 | 0.707107 |
| 50 | 1.00 | 1.414214 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta: 
- Min-Max da un rango de valores entre 0 y 1.
- Toma el valor de 0.5 en Min-Max y 0.0 en Z-Score
- Solamente con Z-Score
- Elegiría Z-Score, ya que al tener un rango más amplio puede haber una mayor variabilidad de los datos y una mejor distribución, al no estar acotado simplemente entre 0 y 1.

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [19]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# TODO: comprobar que las columnas existen en X_train
todas_las_columnas = numericas + nominales + ordinales

# Comprobamos si todas existen en X_train
todas_existen = set(todas_las_columnas).issubset(X_train.columns)
print("¿Existen todas las columnas en X_train?:", todas_existen)
print(X_train[todas_las_columnas])


¿Existen todas las columnas en X_train?: True
     edad  antiguedad_meses  ingresos_anuales  gasto_mensual  num_compras  \
360  45.0              23.0          45801.52          26.93            9   
356  35.0              98.0          56697.53          26.70            9   
138  60.0               8.0          49422.94          88.58            7   
42   23.0              75.0          43333.57          79.68           12   
216  38.0             109.0          29637.21         123.82           10   
..    ...               ...               ...            ...          ...   
109  45.0              75.0          23590.72          28.19           11   
270  44.0             100.0          38668.05          27.97           11   
240  75.0              45.0          42732.28         119.75           10   
66   48.0               NaN          36803.46          55.28            8   
284  25.0              98.0          11516.53          80.64            6   

     devoluciones  satisfacci

**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: 
- Ingresos_anuales y gasto_mensual, ya que son valores númericos que tienen un amplio rango.
- No es obligatorio.
- Que es una variable ordinal/acotada; sus valores van del 1 al 5 por lo que se puede determinar bien el peso al entrenar pero si se usa un algoritmo sensible a distancias si habría que hacer algo.

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [12]:
# En el Pipeline vamos a destacar la lista de pasos que tiene que dar el algoritmo para entrenar al modelo
#  TODO: Describir bien cada paso, ponerle un nombre identificativo con lo que hace
# 1er paso: Imputar --> cargarnos los NaN
# 2º paso: Escalar
pipeline_numerico = Pipeline(steps=[
    ("imputer", imputer_numerico),
    ("scaler", minmax)
])

**¿Por qué imputamos antes de escalar en este pipeline?**

Respuesta: Por que el pipeline es lo que usaremos para entrenar al modelo por lo que no queremos que haya valores inexistentes.

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [13]:
# TODO: 
# Al ser Nominal, imputamos con Moda y escalamos con One-Hot Encoding
pipeline_nominal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", encoder_nominal)
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [14]:
# TODO: 
# Imputamos con Moda y el Encoder será el Ordinal
pipeline_ordinal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", encoder_ordinal)
])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [15]:
# TODO:
# Con ColumnTransformer aplicamos los pipelines
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [16]:
# TODO: 
X_train_preparado = preprocesador.fit_transform(X_train)
print(X_train_preparado[0])
# TODO: fit SOLO SE USA CON EL ENTRENAMIENTO, NUNCA EN TEST
X_test_preparado = preprocesador.transform(X_test)
print(X_test_preparado[0])

[0.47368421 0.1779661  0.4978199  0.08323172 0.46666667 0.2
 0.5        0.         1.         0.         0.         0.
 1.         0.         0.         3.        ]
[0.50877193 0.05084746 0.18356472 0.11300791 0.6        0.6
 0.5        1.         0.         0.         0.         1.
 0.         0.         0.         1.        ]


**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

Respuesta: 
- Porque queremos ver si ha aprendido bien el algoritmo de fit con los datos de train y lo aplica correctamente en los datos de Test.
- Fit aprende a darle los pesos correspondientes a cada variable de las columnas o lo que es lo mismo; aprende los parámetros de transformación de los datos.
- Seguirá ejecutando ya que hemos configurado el handle_unknown='ignore' en el One-Hot-Encoder.
    

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [35]:
# TODO: mostrar X_train_preparado.shape y X_test_preparado.shape
print(X_train_preparado.shape)
print(X_test_preparado.shape)
# TODO: contar np.isnan(...) en ambas matrices
Nan_Train = np.isnan(X_train_preparado).sum()
Nan_Test = np.isnan(X_test_preparado).sum()
print(Nan_Train)
print(Nan_Test)
# TODO: consultar dtype y get_feature_names_out()
print(X_train_preparado.dtype)
print(X_test_preparado.dtype)

print("Tipos de datos después del preprocesado: ", preprocesador.get_feature_names_out())
# TODO: verificar filas de matrices frente a len(y_train), len(y_test)
# Comparativa Entrenamiento
print(X_train_preparado.shape[0])
print(len(y_train))
# Comparativa Test
print(X_test_preparado.shape[0])
print(len(y_test))
# TODO: verificar que entrenamiento y prueba tienen igual número de columnas
print("Cantidad de columnas matriz entrenamiento:",X_train_preparado.shape[1])
print("Cantidad de columnas matriz test:",X_test_preparado.shape[1])

(320, 16)
(80, 16)
0
0
float64
float64
Tipos de datos después del preprocesado:  ['num__edad' 'num__antiguedad_meses' 'num__ingresos_anuales'
 'num__gasto_mensual' 'num__num_compras' 'num__devoluciones'
 'num__satisfaccion' 'nom__dispositivo_Móvil' 'nom__dispositivo_Ordenador'
 'nom__dispositivo_Tablet' 'nom__zona_Centro' 'nom__zona_Este'
 'nom__zona_Norte' 'nom__zona_Oeste' 'nom__zona_Sur'
 'ord__nivel_fidelidad']
320
320
80
80
Cantidad de columnas matriz entrenamiento: 16
Cantidad de columnas matriz test: 16


**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

Respuesta: 
- Debido a que las variables nominales, las ha transformado en matrices/listas para que el modelo entienda esos datos. Por eso el campo dispositivo tiene 3 huecos y zona tiene 5 huecos.
- Sí, es el mismo número de datos características.
- No, ya que después de todo el preprocesamiento con las imputaciones les hemos asignado otro valor.
- No, en cuanto al orden del CSV, pero están en el mismo orden en el cual hemos asignado los pipelines, primero numericas, luego nominales y por ultimo ordinales.

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [42]:
# TODO: columnas_finales = preprocesador.get_feature_names_out()
columnas_finales = preprocesador.get_feature_names_out()
# TODO: df_train_preparado = pd.DataFrame(..., columns=..., index=X_train.index)
df_train_preparado = pd.DataFrame(data=X_train_preparado,columns=columnas_finales,index=X_train.index)

# TODO: df_test_preparado = pd.DataFrame(..., columns=..., index=X_test.index)
df_test_preparado = pd.DataFrame(data=X_test_preparado,columns=columnas_finales,index=X_test.index)

# TODO: mostrar head() de ambos
df_train_preparado.head(5)
df_test_preparado.head(5)

,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
388,0.508772,0.050847,0.183565,0.113008,0.600000,0.6,0.50,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,0.350877,0.788136,0.338105,0.170161,0.400000,0.0,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21,0.982456,0.194915,0.490214,0.207607,0.333333,0.6,0.50,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
368,0.719298,0.177966,0.444791,0.219054,0.133333,0.6,0.75,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
378,0.403509,0.567797,0.164844,0.404752,0.600000,0.0,1.00,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
